# 使用张量表达式生成 TensorIR 代码¶

在许多情况下，我们的开发形式是不在循环级别的更高级别的抽象。 所以另一种常见的获取 TensorIR 的方式是务实地生成相关代码。

张量表达式 (TE) 是一种特定领域的语言，它通过 API 之类的表达式描述一系列计算。

In [2]:
from tvm import te
import tvm
from rich.syntax import Syntax
from rich import print


A = te.placeholder((128, 128), "float32", name="A")
B = te.placeholder((128, 128), "float32", name="B")
k = te.reduce_axis((0, 128), "k")
Y = te.compute((128, 128), lambda i, j: te.sum(A[i, k] * B[k, j], axis=k), name="Y")
C = te.compute((128, 128), lambda i, j: te.max(Y[i, j], 0), name="C")

上面的 lambda 表达式描述了计算 <math xmlns="http://www.w3.org/1998/Math/MathML">
  <msub>
    <mi>Y</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>i</mi>
      <mi>j</mi>
    </mrow>
  </msub>
  <mo>=</mo>
  <munder>
    <mo data-mjx-texclass="OP">&#x2211;</mo>
    <mi>k</mi>
  </munder>
  <msub>
    <mi>A</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>i</mi>
      <mi>k</mi>
    </mrow>
  </msub>
  <msub>
    <mi>B</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>k</mi>
      <mi>j</mi>
    </mrow>
  </msub>
</math>
。在描述计算之后，我们可以通过传递我们感兴趣的相关参数来创建一个 TensorIR 函数。在这种特殊情况下，我们想要创建一个具有两个输入参数（A，B）和一个输出参数（C）的函数。

In [3]:
te_func = te.create_prim_func([A, B, C]).with_attr({"global_symbol": "mm_relu"})
MyModuleFromTE = tvm.IRModule({"mm_relu": te_func})
syntax = Syntax(MyModuleFromTE.script(), "python", theme="monokai", line_numbers=True)
print(syntax)

   1 # from tvm.script import ir as I                                                                              
   2 # from tvm.script import tir as T                                                                             
   3                                                                                                               
   4 @I.ir_module                                                                                                  
   5 class Module:                                                                                                 
   6     @T.prim_func                                                                                              
   7     def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128)
   8         T.func_attr({"tir.noalias": T.bool(True)})                                                            
   9         # with T.block("root"):                                                                               
  10         Y = T.alloc_buffer((128, 128))                                                                        
  11         for i, j, k in T.grid(128, 128, 128):                                                                 
  12             with T.block("Y"):                                                                                
  13                 v_i, v_j, v_k = T.axis.remap("SSR", [i, j, k])                                                
  14                 T.reads(A[v_i, v_k], B[v_k, v_j])                                                             
  15                 T.writes(Y[v_i, v_j])                                                                         
  16                 with T.init():                                                                                
  17                     Y[v_i, v_j] = T.float32(0.0)                                                              
  18                 Y[v_i, v_j] = Y[v_i, v_j] + A[v_i, v_k] * B[v_k, v_j]                                         
  19         for i, j in T.grid(128, 128):                                                                         
  20             with T.block("C"):                                                                                
  21                 v_i, v_j 

0.20.dev0